# 🤝 Multi-Agent Systems: The Supervisor Pattern

## Learning Objectives
In this notebook, you will learn:
1. **Supervisor Architecture** - How a central "supervisor" node can coordinate multiple specialist agents in a LangGraph graph
2. **Structured Routing Decisions** - How to use a Pydantic schema with `with_structured_output` to get reliable routing choices from an LLM
3. **Specialist Agent Nodes** - How to design focused nodes (researcher, writer, critic) that each do one job well
4. **Conditional Routing** - How to wire `add_conditional_edges` so control flows back to the supervisor after each specialist runs
5. **Graph Assembly** - How to compile the individual pieces into a runnable multi-agent `StateGraph`
6. **Loop safety** - A counter reducer (`specialist_calls`) read by the routing function caps the loop, with `recursion_limit` as the backstop and a stub run that proves the cap
7. **Routing traces** - Keep the supervisor's decisions in their own state key (`route_log`), so the transcript holds only the work

## Where this fits
Module 05 sketched the supervisor with hard-coded routing (`03_five_multi_agent_architectures.ipynb`) and applied
it to one workflow (`05_supervisor_workflow_with_human_review.ipynb`). Here the supervisor itself is an LLM that
makes a typed routing decision on every step. `02_hierarchical_agents.ipynb` then stacks supervisors.

## Prerequisites
- Module 05; LangGraph fundamentals (`StateGraph`, nodes, edges, conditional routing)
- Understanding of LangChain message types (`HumanMessage`, `AIMessage`, `SystemMessage`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY`, imports the building blocks, and sets the two loop guards used by every run.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import operator
from typing import Literal

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

# --- Loop guards ------------------------------------------------------------
# This graph can legitimately cycle: the supervisor sends work BACK to the writer
# whenever the critic asks for changes. Two guards stop a never-satisfied critic:
#   MAX_SPECIALIST_CALLS - enforced in code by the routing function, which reads a
#                          counter that every specialist adds 1 to (a clean stop)
#   RECURSION_LIMIT      - LangGraph's hard cap on graph steps (an error, last resort;
#                          the default is 10,007, far past where you want to know)
MAX_SPECIALIST_CALLS = 6
RECURSION_LIMIT = 25
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

### 🤖 Initializing the LLM

All nodes in this graph share a single LLM instance.

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the supervisor and every specialist
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗂️ Part 1: Defining the Shared State

Every LangGraph graph needs a state schema that all nodes read from and write to. For a supervisor-style multi-agent system, the state needs to track the running conversation, which agent should act next, and whether the task is complete — and, kept apart from the conversation, the supervisor's own bookkeeping.

Module 05's `03_five_multi_agent_architectures.ipynb` noted that a router which does not add messages keeps the transcript readable. This state follows that advice: the specialists write to `messages`, the supervisor writes to `route_log`.

### Key Concepts:
- **`messages`**: The work — the task plus every specialist's output, accumulated with `operator.add` so each node can append without overwriting prior turns. No routing chatter.
- **`route_log`**: One entry per supervisor decision (`next` + `reasoning`), also appended with `operator.add`
- **`specialist_calls`**: A counter with an `operator.add` reducer: every specialist returns `1`, LangGraph adds it up, and the routing function reads the total to enforce the loop cap
- **`next_agent`**: Set by the supervisor to tell the graph which specialist to route to next
- **`task_complete` / `final_response`**: Set once the supervisor decides the work is done

In [ ]:
# ============================================================================
# SUPERVISOR STATE: Shared Graph State Schema
# ============================================================================
class SupervisorState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # the work: task + specialist output only
    route_log: Annotated[list[dict], operator.add]  # the supervisor's decisions, one entry each
    specialist_calls: Annotated[int, operator.add]  # counter reducer: each specialist adds 1
    next_agent: str  # the agent that should act next
    task_complete: bool  # whether the task is complete or not
    final_response: str  # the final response to the user when the task is complete


print(f"✅ SupervisorState fields: {list(SupervisorState.__annotations__)}")

---
## 🧭 Part 2: Structured Routing Decisions

The supervisor needs to make a reliable choice between four options: `researcher`, `writer`, `critic`, or `FINISH`. Rather than parsing free-form text, we define a Pydantic schema and bind it to the LLM with `with_structured_output`, so routing decisions come back as validated Python objects.

### `RouteDecision`
The Supervisor's routing decision, expressed as a typed Pydantic model.

In [ ]:
# ============================================================================
# ROUTE DECISION: Structured Output Schema for the Supervisor
# ============================================================================
# Define what the supervisor can decide
class RouteDecision(BaseModel):
    """The Supervisor's routing decision."""

    next: Literal["researcher", "writer", "critic", "FINISH"] = Field(
        description="Which agent to call next, or FINISH if done"
    )
    reasoning: str = Field(description="Why this agent was chosen")


# Create structured output for reliable routing
supervisor_llm = llm.with_structured_output(RouteDecision)

print("✅ Supervisor LLM configured with structured routing output!")

---
## 🧑‍💼 Part 3: The Supervisor Node

The `supervisor` node reads the work so far and decides which specialist should act next, or whether the task is complete. Its prompt spells the routing rules out against the specialists' message tags, so the decision depends on what is in the transcript, not on the model's mood.

> **Key Insight**: the LLM decision is written to state (`next_agent`, `task_complete`, plus one `route_log` entry) and a separate, pure function
> (`route_to_agent`, Part 5) reads it. Routing never calls a model, and the supervisor never writes to `messages`.
> The loop cap lives in that routing function too: it reads the `specialist_calls` counter, not the transcript.

In [ ]:
# ============================================================================
# SUPERVISOR NODE: Decide Which Agent Acts Next
# ============================================================================
def supervisor(state: SupervisorState) -> dict:
    """The Supervisor decides what to do next and logs the decision outside `messages`."""
    system_prompt = """You are a supervisor managing a team of specialists:
    1. researcher - Gathers information and facts
    2. writer - Creates content and text
    3. critic - Reviews and improves work

    Every specialist message starts with its tag: [Researcher], [Writer] or [Critic].
    Decide which agent should act next by applying these rules in order:
    - No [Researcher] message yet -> researcher
    - No [Writer] message yet -> writer
    - The latest message is from [Writer] -> critic
    - The latest message is from [Critic] and says APPROVED -> FINISH
    - The latest message is from [Critic] and asks for changes -> writer
    """

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    decision = supervisor_llm.invoke(messages)

    # The decision goes to state, never into the transcript
    return {
        "next_agent": decision.next,
        "task_complete": decision.next == "FINISH",
        "route_log": [{"next": decision.next, "reasoning": decision.reasoning}],
    }

---
## 👥 Part 4: Specialist Agent Nodes

Each specialist is a focused node with a single responsibility: gather facts, write content, or critique it. All three share the same LLM but use different system prompts. Each also returns `"specialist_calls": 1`; the reducer adds these up, so the loop count never depends on parsing message text.

### 🔬 `researcher`
Gathers information and facts about the original task.

In [ ]:
# ============================================================================
# RESEARCHER NODE: Gathers Information and Facts
# ============================================================================
def researcher(state: SupervisorState) -> dict:
    """Gathers information and facts."""
    system = """You are a research specialist. Your job:
    - Gather relevant facts and information
    - Be thorough but concise
    - Cite sources when possible
    - Focus on what's most useful for the task"""

    # Get the original task from the first human message
    task = next(
        (m.content for m in state["messages"] if isinstance(m, HumanMessage)), ""
    )

    response = llm.invoke(
        [
            SystemMessage(content=system),
            HumanMessage(content=f"Research this topic: {task}"),
        ]
    )
    return {"messages": [AIMessage(content=f"[Researcher] {response.content}")], "specialist_calls": 1}

### ✍️ `writer`
Creates content based on the research (and any critic feedback) available so far.

In [ ]:
# ============================================================================
# WRITER NODE: Creates Content Based on Available Information
# ============================================================================
def writer(state: SupervisorState) -> dict:
    """Creates content based on available information."""
    system = """You are a writing specialist. Your job:
    - Create clear, engaging content
    - Use the research provided
    - Match the requested format and tone
    - If there's critic feedback, incorporate it"""

    # Get recent context (research + any feedback)
    context = "\n".join([m.content for m in state["messages"][-5:]])

    response = llm.invoke(
        [
            SystemMessage(content=system),
            HumanMessage(content=f"Create content based on:\n{context}"),
        ]
    )
    return {"messages": [AIMessage(content=f"[Writer] {response.content}")], "specialist_calls": 1}

### 🧐 `critic`
Reviews the most recent work and either approves it or gives actionable feedback.

In [ ]:
# ============================================================================
# CRITIC NODE: Reviews Work and Provides Feedback
# ============================================================================
def critic(state: SupervisorState) -> dict:
    """Reviews work and provides feedback."""
    system = """You are a quality critic. Your job:
    - Review the content objectively
    - Provide specific, actionable feedback
    - If the work is good, say "APPROVED" and explain why
    - If it needs work, explain exactly what to improve"""

    # Get the most recent work
    context = "\n".join([m.content for m in state["messages"][-3:]])

    response = llm.invoke(
        [
            SystemMessage(content=system),
            HumanMessage(content=f"Review this work:\n{context}"),
        ]
    )
    return {"messages": [AIMessage(content=f"[Critic] {response.content}")], "specialist_calls": 1}

---
## 🏁 Part 5: Finalizing and Routing Logic

Two small helper functions close the loop: one extracts the final answer once the supervisor marks the task complete, and one translates the supervisor's decision into a graph edge target.

### `finalize`
Extracts the final response (the last Writer output) once the task is complete.

In [ ]:
# ============================================================================
# FINALIZE NODE: Extract the Final Response
# ============================================================================
def finalize(state: SupervisorState) -> dict:
    """Extract the final response when the task is complete (or the cap stopped the loop)."""
    # Find the last Writer output
    for msg in reversed(state["messages"]):
        if isinstance(msg, AIMessage) and "[Writer]" in msg.content:
            content = msg.content.replace("[Writer] ", "")
            return {"final_response": content}
    return {"final_response": "Task completed."}

### `route_to_agent`
Routes the graph based on the Supervisor's decision - to a specialist, or to `finalize` when the task is done or the cap is reached. It is a pure function: it reads state and calls no model, so the cap holds whatever the model decides.

In [ ]:
# ============================================================================
# ROUTING FUNCTION: Route Based on Supervisor's Decision, Never Past the Cap
# ============================================================================
def route_to_agent(state: SupervisorState) -> str:
    """Route on the Supervisor's decision, but never past MAX_SPECIALIST_CALLS."""
    # Code-enforced cap: the counter wins over whatever the model decided
    if state["specialist_calls"] >= MAX_SPECIALIST_CALLS:
        return "finalize"
    if state["task_complete"]:
        return "finalize"
    return state["next_agent"]

---
## 🕸️ Part 6: Assembling the Multi-Agent Graph

With every node and the routing function defined, we can wire them into a `StateGraph`: the supervisor always runs first, routes to a specialist via conditional edges, and every specialist routes back to the supervisor until it decides to `FINISH`.

Every node is a parameter that defaults to the real one, so Part 9 can swap in stubs and test the loop cap without calling a model.

### `build_multi_agent_system`
Builds and compiles the complete multi-agent graph.

In [ ]:
# ============================================================================
# GRAPH ASSEMBLY: Build the Complete Multi-Agent System
# ============================================================================
def build_multi_agent_system(
    supervisor_node=supervisor, researcher_node=researcher, writer_node=writer, critic_node=critic
):
    """Build the complete multi-agent graph. Each node is a parameter, so a stub can replace it."""
    # Create graph
    graph = StateGraph(SupervisorState)

    # Add all nodes
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("researcher", researcher_node)
    graph.add_node("writer", writer_node)
    graph.add_node("critic", critic_node)
    graph.add_node("finalize", finalize)

    # Entry point: always start with Supervisor
    graph.add_edge(START, "supervisor")

    # Supervisor routes to specialists
    graph.add_conditional_edges(
        "supervisor",
        route_to_agent,
        {
            "researcher": "researcher",
            "writer": "writer",
            "critic": "critic",
            "finalize": "finalize",
        },
    )

    # After each specialist, go back to Supervisor
    graph.add_edge("researcher", "supervisor")
    graph.add_edge("writer", "supervisor")
    graph.add_edge("critic", "supervisor")

    # Finalize ends the graph
    graph.add_edge("finalize", END)

    return graph.compile()


print("✅ build_multi_agent_system() defined!")

In [ ]:
# ============================================================================
# BUILD AND INSPECT THE GRAPH
# ============================================================================
multi_agent_graph = build_multi_agent_system()
for edge in multi_agent_graph.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

In [ ]:
multi_agent_graph

---
## ▶️ Part 7: Running the Multi-Agent System

The cell seeds a task, runs the graph with both loop guards, and prints the transcript (`messages`) and the routing log (`route_log`) separately, then the final output. The ✅ line checks in code that no routing message reached the transcript; the line above it says whether the supervisor chose `FINISH` or the cap ended the run.

In [ ]:
# ============================================================================
# RUN: Build the Graph and Execute a Sample Task
# ============================================================================
agent = multi_agent_graph
SPECIALIST_TAGS = ("[Researcher]", "[Writer]", "[Critic]")


def new_task(task: str) -> dict:
    """Initial state: the task is the only message; the log and the counter start empty."""
    return {
        "messages": [HumanMessage(content=task)],
        "route_log": [],
        "specialist_calls": 0,
        "next_agent": "",
        "task_complete": False,
        "final_response": "",
    }


print("=" * 60)
print("MULTI-AGENT SYSTEM")
print("=" * 60)

# Run the system, capped so a never-satisfied critic cannot loop forever.
try:
    result = agent.invoke(
        new_task("Write a short blog post about the benefits of AI in healthcare"),
        config={"recursion_limit": RECURSION_LIMIT},
    )
except GraphRecursionError:
    print()
    print(f"⚠️  Stopped after {RECURSION_LIMIT} steps without finishing.")
    print("   The Supervisor kept routing and never chose FINISH — usually")
    print("   because the critic never says APPROVED, so work bounces")
    print("   between writer and critic forever.")
    print()
    print("   Fix the routing prompt, or raise RECURSION_LIMIT if the task")
    print("   genuinely needs more revision rounds.")
    raise

# The transcript: only the task and the specialists' work
print("\nAgent Conversation (messages):")
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        # Truncate for display
        content = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
        print(f"\n{content}")

# The supervisor's decisions, read from their own key
print("\nRouting log (route_log):")
for i, step in enumerate(result["route_log"], 1):
    print(f"  {i}. → {step['next']}: {step['reasoning']}")

ended_by = (
    "FINISH" if result["route_log"][-1]["next"] == "FINISH"
    else f"the cap ({MAX_SPECIALIST_CALLS} specialist calls)"
)
print(f"\nSpecialist calls: {result['specialist_calls']} · run ended by {ended_by}")
transcript_clean = all(
    isinstance(m, HumanMessage) or m.content.startswith(SPECIALIST_TAGS) for m in result["messages"]
)
print(("✅" if transcript_clean else "❌") + " messages hold only the task and specialist output; routing lives in route_log")

print("\n" + "=" * 60)
print("FINAL OUTPUT:")
print("=" * 60)
print(result["final_response"])

---
## 🧭 Part 8: Reading the Supervisor's Routing Trace

The supervisor writes one `route_log` entry per decision, so the routing path — and the reason for each hop — can be read straight out of the result, without filtering the transcript. A short task (a tagline) usually needs fewer hops than the blog post; the cell prints both counts.

In [ ]:
# ============================================================================
# PART 8: ROUTING DECISIONS ONLY
# ============================================================================
trace_result = multi_agent_graph.invoke(
    new_task("Create a marketing tagline for a new coffee brand"),
    config={"recursion_limit": RECURSION_LIMIT},
)

print("Routing decisions:")
for step in trace_result["route_log"]:
    print(f"  → {step['next']}: {step['reasoning']}")
print(f"\nFinal tagline: {trace_result['final_response']}")
print(f"\nSpecialist calls — tagline: {trace_result['specialist_calls']}, blog post: {result['specialist_calls']}")

---
## 🛡️ Part 9: Proving the Loop Cap

A real critic usually approves within a round or two, so the cap rarely fires in Part 7. To see it work, rebuild the same graph with two stubs: a supervisor that never says `FINISH` (it always sends the work back to the writer) and a writer that returns a numbered draft without calling a model. Without the cap, only `recursion_limit` would stop this graph, with an error instead of a result. With it, `route_to_agent` sends the run to `finalize` once the counter reaches `MAX_SPECIALIST_CALLS`. The run makes no LLM calls.

In [ ]:
# ============================================================================
# PROVING THE CAP: A Supervisor That Never Says FINISH (no LLM calls)
# ============================================================================
def stuck_supervisor(state: SupervisorState) -> dict:
    """Stand-in for a supervisor that is never satisfied: always sends the work back to the writer."""
    return {
        "next_agent": "writer",
        "task_complete": False,
        "route_log": [{"next": "writer", "reasoning": "stub: never satisfied"}],
    }


def stub_writer(state: SupervisorState) -> dict:
    """Stand-in writer: returns a numbered draft without calling a model."""
    return {
        "messages": [AIMessage(content=f"[Writer] draft {state['specialist_calls'] + 1}")],
        "specialist_calls": 1,
    }


stuck_graph = build_multi_agent_system(supervisor_node=stuck_supervisor, writer_node=stub_writer)
capped = stuck_graph.invoke(new_task("Write anything"), config={"recursion_limit": RECURSION_LIMIT})

decisions = [step["next"] for step in capped["route_log"]]
print(f"Supervisor decisions ({len(decisions)}): {decisions}")
print(f"Specialist calls: {capped['specialist_calls']} (cap = {MAX_SPECIALIST_CALLS})")
print(f"Final response:   {capped['final_response']}")

stopped_by_cap = capped["specialist_calls"] == MAX_SPECIALIST_CALLS and "FINISH" not in decisions
print(("✅" if stopped_by_cap else "❌") + " the cap ended a run whose supervisor never chose FINISH")

---
## 📝 Summary

In this notebook, we built a supervisor-style multi-agent system with LangGraph.

### 1. State and Routing
- **`SupervisorState`**: Shared graph state — `messages` holds only the work, `route_log` holds the supervisor's decisions, and `specialist_calls` counts specialist turns with an `operator.add` reducer
- **`RouteDecision`**: A Pydantic schema used with `with_structured_output` for reliable supervisor routing
- **`route_to_agent`**: A pure function that translates the supervisor's decision into a graph edge target, and enforces the cap

### 2. Nodes
- **`supervisor`**: Decides which specialist should act next, or whether to finish, and logs the decision in `route_log`
- **`researcher`**, **`writer`**, **`critic`**: Focused specialist nodes, each with a single responsibility; each adds 1 to `specialist_calls`
- **`finalize`**: Extracts the final response once the supervisor marks the task complete or the cap stops the loop

### 3. Graph Assembly
- **`build_multi_agent_system`**: Wires all nodes together - supervisor routes out via conditional edges, and every specialist routes back to the supervisor until `FINISH`. Every node is a parameter, so a stub can replace any of them

### 4. Loop safety and traces
- **`MAX_SPECIALIST_CALLS`** stops the loop cleanly: `route_to_agent` reads the counter, never the transcript, and a stub supervisor that never says `FINISH` proves it (Part 9). **`recursion_limit`** is the hard backstop
- **`route_log`** is a readable routing trace with a reason for every hop, and it keeps routing chatter out of the transcript

### Next Steps
- **`02_hierarchical_agents.ipynb`**: supervisors of supervisors — a CEO supervisor chains department subgraphs, and one department runs its own supervisor